# Restricted-input method reference
Source: gps_analysis/residence_identification/02-residence_calibration.ipynb
Requires independently obtained inputs and source directory layout. Not an end-to-end pipeline. Outputs can contain sensitive data; keep the working directory private. External absolute paths have been replaced by external_inputs/<basename>. See docs/REPRODUCIBILITY.md.


In [ ]:
import os
from pathlib import Path
private_root = Path(os.environ["UFUG_PRIVATE_ROOT"]).resolve()
os.chdir(private_root)


In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.ops import nearest_points

import os
# Working directory is explicitly configured in the first cell.


In [ ]:
housing_boundary_sg = gpd.read_file('external_inputs/housing_boundary_singapore.geojson')

In [ ]:
def get_nearest_point(row):
    pt = row.geometry
    poly = housing_boundary_sg.loc[row["index_right"], "geometry"]
    _, nearest_on_poly = nearest_points(pt, poly)
    return nearest_on_poly

year = "2022"

for park in ['jurong', 'amk', 'punggol']:
    residences_gdf = gpd.read_file(f"gps_analysis/activities_identification_example/predicted_residence_{park}_{year}.geojson")

    residences_gdf = residences_gdf[~residences_gdf["geometry"].isna()]

    residences_gdf = residences_gdf.to_crs(housing_boundary_sg.crs)

    calibrated_points = gpd.sjoin_nearest(
        residences_gdf,
        housing_boundary_sg,
        how="left",
        distance_col="dist_to_nearest_poly"
    )

    # Create a geometry column for the nearest point on the polygon
    calibrated_points["nearest_geom"] = calibrated_points.apply(get_nearest_point, axis=1)
    
    calibrated_points["geometry"] = calibrated_points["nearest_geom"]
    calibrated_points = calibrated_points.set_geometry("geometry")

    # Extract coordinates
    calibrated_points["nearest_x"] = calibrated_points["nearest_geom"].x
    calibrated_points["nearest_y"] = calibrated_points["nearest_geom"].y

    # Drop helper geometry
    calibrated_points = calibrated_points.drop(columns=["nearest_geom"])

    calibrated_points.to_file(f"gps_analysis/activities_identification_example/calibrated_residence_{park}_{year}_updated.geojson", driver='GeoJSON')

In [ ]:
calibrated_points